# 03 · Stage 2 · LLM 微调（表格 QA）

**模型**：`mlx-community/Qwen2.5-0.5B-Instruct-4bit` (~400MB)

**速度**：M2 Pro 约 10–15 分钟 2000 样本 2 epoch。

In [1]:
# ⚠️ 前置检查：train.jsonl 不存在就自动帮你跑 prepare_stage2.py
import subprocess, sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from src import config as C

s2 = C.DATA_DIR / 'stage2_train' / 'train.jsonl'
raw_fin = C.RAW_DIR / 'fintabnet_sample'
raw_pub = C.RAW_DIR / 'pubtabnet_sample'
if not s2.exists():
    if not (raw_fin.exists() or raw_pub.exists()):
        raise RuntimeError(
            '先跑 `uv run python scripts/download_data.py` 下载数据集'
        )
    print('⏳ stage2_train/train.jsonl 不存在，自动跑 prepare_stage2.py ...')
    subprocess.check_call([sys.executable, str(C.ROOT / 'scripts' / 'prepare_stage2.py')])
print('✅ 就绪:', s2)

✅ 就绪: /Users/pc-rn/ws/ai-lab/ocr-fine-app/data/stage2_train/train.jsonl


In [2]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from src import config as C
from src.data import load_jsonl


## 1. 转成 MLX-LM 格式

`mlx_lm.lora` 对 chat 模型期望：`{"messages": [{"role":..., "content":...}]}`

In [3]:
def alpaca_to_chat(r):
    user = f"{r['instruction']}\n\n{r['input']}"
    return {'messages': [
        {'role': 'user', 'content': user},
        {'role': 'assistant', 'content': r['output']},
    ]}

for name in ['train', 'val']:
    rows = [alpaca_to_chat(r) for r in load_jsonl(C.DATA_DIR/'stage2_train'/f'{name}.jsonl')]
    out = C.DATA_DIR / 'stage2_mlx'
    out.mkdir(parents=True, exist_ok=True)
    with (out / (f'{"valid" if name=="val" else name}.jsonl')).open('w', encoding='utf-8') as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False)+'\n')
print('done')

done


## 2. 启动 LoRA 训练

```bash
python -m mlx_lm.lora \
    --model mlx-community/Qwen2.5-0.5B-Instruct-4bit \
    --train \
    --data data/stage2_mlx \
    --iters 600 \
    --batch-size 2 \
    --num-layers 8 \
    --learning-rate 2e-4 \
    --adapter-path models/stage2_adapter
```

## 3. 推理对比

In [4]:
from src.infer import chat

table_md = "| 年份 | 营收(亿) | 净利润(亿) |\n|---|---|---|\n| 2022 | 100 | 15 |\n| 2023 | 120 | 18 |\n| 2024 | 135 | 22 |"
q = '哪一年净利润同比增长最大？'
msgs = [{'role': 'system', 'content': '你是表格问答助手'},
        {'role': 'user', 'content': f'表格:\n{table_md}\n问题:{q}'}]
print('=== base ===')
print(chat(msgs, adapter=None))
print('\n=== finetuned ===')
print(chat(msgs, adapter=str(C.STAGE2_ADAPTER)))

=== base ===


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

2024年净利润同比增长最大，同比增长率是22%。

=== finetuned ===


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

FileNotFoundError: The adapter path does not exist: /Users/pc-rn/ws/ai-lab/ocr-fine-app/models/stage2_adapter

## 4. 批量评估

In [ ]:
from src.eval import exact_match, token_f1, rouge_l
from statistics import mean

val = load_jsonl(C.DATA_DIR/'stage2_train'/'val.jsonl')[:30]
ems, f1s, rs = [], [], []
for r in val:
    msgs = [{'role':'user', 'content': f"{r['instruction']}\n\n{r['input']}"}]
    pred = chat(msgs, adapter=str(C.STAGE2_ADAPTER), max_tokens=256)
    ems.append(exact_match(pred, r['output']))
    f1s.append(token_f1(pred, r['output']))
    rs.append(rouge_l(pred, r['output']))
print(f'EM={mean(ems):.3f}  F1={mean(f1s):.3f}  ROUGE-L={mean(rs):.3f}')